# 03 · Fuentes públicas y linajes del Marco Geográfico Electoral

**Pregunta.** ¿Qué continuidad territorial puede defenderse usando sólo información pública y cómo debe clasificarse cada sección antes de modelar?

**Entradas.** El libro público AMGE 2005–2022 publicado por DERFE mediante una resolución de transparencia existente, el agregado limpio sección–elección y configuraciones versionadas.

**Método.** Se valida el SHA-256 del libro, se interpretan por separado secciones origen y destino, se expanden rangos únicamente cuando existe guion y se crean aristas sólo para divisiones o fusiones completas uno-a-muchos o muchos-a-uno. Otros cambios se conservan como ambiguos.

**Disponibilidad temporal.** Para evaluar un target en el año `t` sólo se permiten eventos con `effective_year < t`. Esto evita incorporar retrospectivamente modificaciones del mismo año cuya fecha exacta no esté disponible.

**Límite.** El archivo termina en 2022. La clasificación 2024 es útil pero parcial; no se declara completa la evidencia de cambios 2023. No se construye contigüidad espacial ni se distribuyen conteos históricos entre descendientes.

**Salidas.** Manifiesto público, eventos, secciones afectadas, aristas de linaje, componentes y clases A/B/C/D por elección.

## Entorno y contratos

Se comprueban configuraciones, libro y panel previo. La fuente, reglas de elegibilidad y categorías se muestran antes de ejecutar transformaciones.

In [ ]:
import time
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import Markdown, display


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "configs/public_geography.yaml").exists():
            return candidate
    raise FileNotFoundError("No se encontró la raíz del proyecto")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
RUN_STARTED = time.perf_counter()
required = [
    PROJECT_ROOT / "configs/public_geography.yaml",
    PROJECT_ROOT / "configs/evaluation.yaml",
    PROJECT_ROOT / "data/interim/public_ine/amge_2005_2022.xlsx",
    PROJECT_ROOT / "artifacts/tables/provisional_section_year_totals.parquet",
]
checks = {path.relative_to(PROJECT_ROOT).as_posix(): path.exists() for path in required}
display(pd.Series(checks, name="existe").to_frame())
assert all(checks.values())

In [ ]:
with (PROJECT_ROOT / "configs/public_geography.yaml").open(encoding="utf-8") as stream:
    public_geo_config = yaml.safe_load(stream)
with (PROJECT_ROOT / "configs/evaluation.yaml").open(encoding="utf-8") as stream:
    evaluation_config = yaml.safe_load(stream)

display(pd.DataFrame([
    {"source_id": source_id, **source}
    for source_id, source in public_geo_config["sources"].items()
]))
display(pd.DataFrame([
    {"class": class_id, **definition}
    for class_id, definition in evaluation_config["confidence_classes"].items()
]))

In [ ]:
display(Markdown(
    "**Interpretación.** AMGE es la única fuente usada para construir aristas. El MGS 2022, "
    "Tableau y ECEG 2020 están registrados, pero no se convierten silenciosamente en crosswalks: "
    "sirven como referencia cartográfica o futuras covariables."
))

## Procedencia y validación del libro AMGE

El manifiesto registra autoridad, resolución oficial, enlace público, ruta local, bytes y hash. La transformación falla si el hash difiere del valor congelado.

In [ ]:
from radar27.geography.amge import write_public_source_manifest

public_manifest = write_public_source_manifest(PROJECT_ROOT)
display(public_manifest)

In [ ]:
amge_source = public_manifest[public_manifest.source_id.eq("amge_history")].iloc[0]
display(Markdown(
    f"**Resultado.** El libro AMGE tiene `{int(amge_source.bytes):,}` bytes y SHA-256 "
    f"`{amge_source.sha256}`. Fue obtenido de una carpeta pública enlazada por la resolución "
    "oficial, sin autenticación ni nueva solicitud de información."
))

## Eventos, afectados y aristas elegibles

Un guion representa rango inclusivo; coma o la conjunción “Y” representan códigos discretos. Se crean aristas sólo si el caso normalizado es división/fusión, existen ambos lados, hay año efectivo y la forma es uno-a-muchos o muchos-a-uno.

In [ ]:
from radar27.geography.amge import parse_amge_workbook

events, edges, affected = parse_amge_workbook(PROJECT_ROOT)
event_summary = (
    events.groupby(["effective_year", "lineage_eligible"], dropna=False, as_index=False)
    .agg(event_rows=("event_id", "count"), origins=("origin_count", "sum"), destinations=("destination_count", "sum"))
)
display(event_summary)
display(events[[
    "event_id", "source_excel_row", "entity_id", "program", "case_label", "effective_year",
    "origin_count", "destination_count", "mapping_shape", "lineage_eligible",
]].head(20))

In [ ]:
display(Markdown(
    f"**Resultado.** Se preservaron `{len(events):,}` filas-evento, `{len(affected):,}` "
    f"registros de sección afectada y `{len(edges):,}` aristas elegibles. "
    f"`{int(events.lineage_eligible.sum()):,}` filas-evento permiten linaje; las restantes "
    "quedan como evidencia ambigua y nunca generan una asignación ponderada."
))

## Componentes y clases de confianza

Las aristas elegibles forman componentes conexos no dirigidos. Una sección pasa a D si ella o cualquier miembro de su componente tiene un evento ambiguo previo. La clasificación se recalcula para cada corte temporal.

In [ ]:
from radar27.modeling.lineage_evaluation import build_confidence_classes

classes, components = build_confidence_classes(PROJECT_ROOT, affected, edges)
class_summary = (
    classes.groupby(["target_year", "geography_confidence_class"], as_index=False)
    .agg(sections=("section_key_observed", "nunique"), nominal_list=("nominal_list", "sum"))
)
year_totals = class_summary.groupby("target_year").agg(
    total_sections=("sections", "sum"), total_nominal_list=("nominal_list", "sum")
)
class_summary = class_summary.join(year_totals, on="target_year")
class_summary["section_share"] = class_summary.sections / class_summary.total_sections
class_summary["nominal_list_share"] = class_summary.nominal_list / class_summary.total_nominal_list
display(class_summary)

In [ ]:
current = class_summary[class_summary.target_year.eq(2024)]
display(Markdown(
    "**Resultado 2024.** " + "; ".join(
        f"clase {row.geography_confidence_class}: {int(row.sections):,} secciones "
        f"({row.section_share:.1%})"
        for row in current.itertuples(index=False)
    ) + ". La cobertura AMGE termina en 2022, por lo que estas clases no absorben todavía "
    "las relaciones aprobadas durante 2023."
))

## Comprobaciones de cierre

Se exige unicidad de eventos/aristas, años dentro de cobertura, componentes deterministas y clasificación exhaustiva.

In [ ]:
expected_classes = {"A", "B", "C", "D"}
close_checks = {
    "amge_hash_verified": bool(amge_source.local_exists),
    "event_ids_unique": events.event_id.is_unique,
    "edges_unique": not edges.duplicated().any(),
    "eligible_edges_have_two_sides": bool(edges.origin_section_key.notna().all() and edges.destination_section_key.notna().all()),
    "classification_exhaustive": set(classes.geography_confidence_class) == expected_classes,
    "one_class_per_target_section": not classes.duplicated(["target_year", "section_key_observed"]).any(),
    "amge_2024_coverage_complete": False,
    "fractional_allocation_used": False,
}
display(pd.Series(close_checks, name="valor").to_frame())
display(Markdown(
    f"**Cierre.** Se procesaron `{len(events):,}` eventos en `{time.perf_counter()-RUN_STARTED:.1f}` s. "
    "La cobertura 2024 permanece deliberadamente incompleta y la asignación fraccionaria permanece "
    "falsa. El notebook 04 evalúa las tres estrategias con estas restricciones visibles."
))